In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../src")

In [3]:
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf
from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import (
    LinguisticExpressionExtractionLayer,
)
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend
from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer

pdf_path = "../data/XQuality/Textual/Chapitre_8_Alarmes_et_messages.pdf"
model_name = "gemma2:27b-instruct-q4_K_M"

raw_text = extract_text_from_pdf(pdf_path)

document = Document(
    doc_id="pdf_test_001",
    source_path=pdf_path,
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level="Treat machine types, failure types, and symptom types as concepts; treat document-specific occurrences as individuals.",
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=model_name,
    user_guidance=guidance,
)

ollama = OllamaBackend()

translator = DeepTranslatorBackend(max_chars=4000)

pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=True,          # enable translation
            translator=translator,   # use free translator
            source_language="fr",    # for example
            target_language="en",
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=2,
            temperature=0.0,
        ),
    ]
)


runner = Runner(pipeline)
final_state = runner.run(state)

final_state.logs

SyntaxError: from __future__ imports must occur at the beginning of the file (linguistic_expression.py, line 30)

In [5]:
len(final_state.document.chunks)

27

In [ ]:
for expr in final_state.linguistic_expressions[:10]:
    print(expr)

LinguisticExpression(expr_id='expr_00000', text='alarms', label='failure symptom', justification='', evidence=[Evidence(chunk_id='chunk_0000', snippet='SPRINT32linear-FA-MP-v0.1-en Page 8-4 8 Alarms and messages 8.1 Introduction Alarms and messages are divided into two main categories: 1 alarms and messages from the CNC, for which we refer to the maintenance manual of the Fanuc series CNC........ 2 alarms and messages from the PLC interface, descri')], confidence=None)
LinguisticExpression(expr_id='expr_00001', text='messages', label='failure symptom', justification='', evidence=[Evidence(chunk_id='chunk_0000', snippet='SPRINT32linear-FA-MP-v0.1-en Page 8-4 8 Alarms and messages 8.1 Introduction Alarms and messages are divided into two main categories: 1 alarms and messages from the CNC, for which we refer to the maintenance manual of the Fanuc series CNC........ 2 alarms and messages from the PLC interface, descri')], confidence=None)
LinguisticExpression(expr_id='expr_00002', text='i